# R2AI2026 — sinh `pandas_query` trên Kaggle (T4 GPU)

Điều kiện trước khi chạy:
1. Bật **GPU T4** (Settings → Accelerator) và **Internet** (để `git clone` + tải model).
2. Upload `retrieval_results.jsonl` (build ở local bằng `python -m r2ai.retrieval.run_retrieval`) làm **Kaggle Dataset**, rồi Add Data vào notebook. File này đã nhúng sẵn CSV của các bảng candidate nên **không cần mount corpus 362MB**.
3. Sửa `REPO_URL` và `RETRIEVAL_PATH` bên dưới cho khớp.

Output: `predictions.jsonl` ghi **append + flush sau mỗi câu** trong `/kaggle/working` — tải về rồi chạy `python -m r2ai.packaging.assemble_submission` ở local (re-execute lại toàn bộ query trước khi đóng gói).

In [ ]:
REPO_URL = "https://github.com/CryAndRRich/r2ai-stage2.git"
RETRIEVAL_PATH = "/kaggle/input/datasets/nhtquyn/r2ai2026/retrieval_results.jsonl"
PREDICTIONS_PATH = "/kaggle/working/predictions.jsonl"
WORK_DIR = "/kaggle/working/exec"
PILOT_N = 20  # chạy thử trước khi chạy full 1.012 câu

In [ ]:
!git clone --depth 1 $REPO_URL /kaggle/working/r2ai-stage2
%cd /kaggle/working/r2ai-stage2
!pip install -q -r requirements.txt

In [ ]:
import sys

sys.path.insert(0, "/kaggle/working/r2ai-stage2")
import torch

print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
assert torch.cuda.is_available(), (
    "Không phát hiện GPU! Vào Settings (panel bên phải notebook) -> Accelerator -> chọn GPU T4 x2 "
    "(hoặc T4 x1) -> Save -> notebook sẽ restart session. Chạy lại từ đầu sau đó. Không có GPU thì "
    "load model 7B sẽ cực chậm/OOM trên CPU."
)

# Fail sớm ngay tại đây nếu cell cài đặt ở trên bị lỗi, thay vì để lỗi rơi xuống tận lúc load model 7B.
import bitsandbytes
import numpy
import pandas
import transformers

print("pandas", pandas.__version__, "| numpy", numpy.__version__,
      "| transformers", transformers.__version__, "| bitsandbytes", bitsandbytes.__version__)

In [ ]:
# Đăng nhập HuggingFace Hub (không bắt buộc với Qwen, chỉ để bớt cảnh báo rate-limit khi tải model).
# Điền token thật vào đây SAU KHI đã import notebook này lên Kaggle — đừng commit bản đã điền token
# ngược lại về git. Bọc try/except: chạy "Save and Run All" không có người canh, token còn để
# placeholder hoặc sai thì KHÔNG được làm dừng cả notebook (Qwen tải được không cần token).
from huggingface_hub import login

try:
    login("HF_TOKEN")  # TODO: dán token HuggingFace thật vào đây trên Kaggle
    print("Đã đăng nhập HuggingFace Hub.")
except Exception as exc:
    print(f"Bỏ qua đăng nhập HF ({exc}) — vẫn chạy tiếp không token, chỉ bị giới hạn rate-limit.")

## 1. Smoke test: prompt + sandbox (không cần GPU)

`--dry-run` không nạp LLM: chỉ dựng prompt, ghi CSV ra đĩa và chạy sandbox. Bắt sớm lỗi encode CSV / đường dẫn trước khi tốn thời gian GPU.

In [ ]:
import subprocess

proc = subprocess.run([
    "python", "-m", "r2ai.generation.run_generation",
    "--retrieval", RETRIEVAL_PATH, "--out", "/kaggle/working/predictions_dryrun.jsonl",
    "--work-dir", WORK_DIR, "--limit", "3", "--dry-run", "--no-resume",
])
assert proc.returncode == 0, (
    f"Smoke test thất bại (exit code {proc.returncode}) — xem traceback phía trên. Chạy 'Save and "
    "Run All' sẽ dừng ở đây, không tốn thời gian chạy tiếp pilot/full khi plumbing cơ bản đã hỏng."
)

## 2. Pilot 20 câu — đo wall-clock/câu

Ước lượng tổng thời gian cho 1.012 câu so với giới hạn session Kaggle (~9-12h). Nếu quá lâu: đổi sang `Qwen/Qwen2.5-Coder-3B-Instruct` (`--model`) hoặc giảm `candidates_in_prompt` trong `configs/baseline.yaml`.

Chạy "Save and Run All": pilot thất bại hoặc `exec_ok=0` sẽ **dừng notebook tại đây**, không chạy tiếp bước full 1.012 câu (12h) trên một pipeline đã biết hỏng.

In [ ]:
import json
import subprocess
import time

start = time.time()
proc = subprocess.run([
    "python", "-m", "r2ai.generation.run_generation",
    "--retrieval", RETRIEVAL_PATH, "--out", PREDICTIONS_PATH, "--work-dir", WORK_DIR,
    "--limit", str(PILOT_N),
])
elapsed = time.time() - start

rows = [json.loads(line) for line in open(PREDICTIONS_PATH, encoding="utf-8") if line.strip()]
exec_ok = sum(1 for r in rows if r["exec_ok"])
assert proc.returncode == 0 and exec_ok > 0, (
    f"Pilot THẤT BẠI (exit code {proc.returncode}, exec_ok={exec_ok}/{len(rows)}) — sửa lỗi ở "
    "traceback/predictions.jsonl phía trên rồi chạy lại. 'Save and Run All' dừng tại đây, không "
    "chạy tiếp bước full 1.012 câu (~nhiều giờ) trên một pipeline đã biết hỏng."
)
print(f"exec_ok pilot: {exec_ok}/{len(rows)}")
print(f"{elapsed / PILOT_N:.1f}s/câu -> ước tính {elapsed / PILOT_N * 1012 / 3600:.1f}h cho 1.012 câu")

## 3. Chạy full (resume được)

Mặc định `--resume`: bỏ qua các id đã có trong `predictions.jsonl`, nên chạy lại cell này sau khi session bị ngắt là tiếp tục từ chỗ dừng.

In [ ]:
!python -m r2ai.generation.run_generation \
    --retrieval $RETRIEVAL_PATH --out $PREDICTIONS_PATH --work-dir $WORK_DIR

In [ ]:
import json
from collections import Counter

rows = [json.loads(line) for line in open(PREDICTIONS_PATH, encoding="utf-8") if line.strip()]
print("tổng:", len(rows), "| id duy nhất:", len({r["id"] for r in rows}))
print("exec_ok:", Counter(r["exec_ok"] for r in rows))
for row in [r for r in rows if not r["exec_ok"]][:5]:
    print(row["id"], "->", (row["exec_error"] or "")[:200])

Tải `predictions.jsonl` về máy local, đặt vào `data/interim/`, rồi:

```bash
python -m r2ai.packaging.assemble_submission   # join + re-execute ở local
python -m r2ai.packaging.zip_submission        # validate + zip
```